# 🧹 Tahap 2: Preprocessing – Data Cleaning

Notebook ini menjalankan **full preprocessing pipeline** pada data teks mentah Bahasa Indonesia.

**Tahapan cleaning:**
1. Case folding (lowercase)
2. Hapus URL, mention (@), hashtag (#)
3. Hapus emoji & karakter non-ASCII
4. Hapus tanda baca & angka
5. Normalisasi kata tidak baku (kamus manual + `kamuskatabaku.xlsx`)
6. Stopword removal (menjaga kata negasi)
7. Stemming (Sastrawi)

**Input:**
- `data/raw/DatasetSentimen.csv`
- `data/raw/kamuskatabaku.xlsx`

**Output:** `data/processed/dataset_clean_pipeline.csv`

> ⚠️ **Catatan:** Untuk pipeline modeling selanjutnya digunakan `dataset_clean_manual.csv`
> (sudah berlabel manual). File `dataset_clean_pipeline.csv` hanya sebagai referensi.

In [1]:
# ── Setup Path ──────────────────────────────────────────────────────────────
import os
import sys

def _find_root(start):
    """Cari root directory project (berisi folder 'src' dan 'data')."""
    d = start
    for _ in range(5):
        if os.path.isdir(os.path.join(d, 'src')) and os.path.isdir(os.path.join(d, 'data')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return start

ROOT_DIR   = _find_root(os.path.abspath(os.getcwd()))
SRC_DIR    = os.path.join(ROOT_DIR, 'src')
OUTPUT_DIR = os.path.join(ROOT_DIR, 'output')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'ROOT_DIR   : {ROOT_DIR}')
print(f'SRC_DIR    : {SRC_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')

ROOT_DIR   : D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3
SRC_DIR    : D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\src
OUTPUT_DIR : D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\output


In [2]:
# ── Import ──────────────────────────────────────────────────────────────────
from data_loader import load_raw_data, load_kamus, save_processed_data
from preprocessing import (
    setup_stemmer,
    load_stopwords_github,
    build_stopword_final,
    build_kamus_gabungan,
    run_preprocessing
)

In [3]:
# ── Load Data Raw ────────────────────────────────────────────────────────────
df_raw      = load_raw_data('DatasetSentimen.csv')
kamus_excel = load_kamus('kamuskatabaku.xlsx')

print(f'\nJumlah data raw  : {len(df_raw):,} baris')
print(f'Kolom            : {df_raw.columns.tolist()}')
df_raw.head()

[data_loader] Dataset 'DatasetSentimen.csv' dimuat: 11163 baris, 1 kolom.


[data_loader] Kamus 'kamuskatabaku.xlsx' dimuat: 15185 entri.
             Kolom: ['tidak_baku', 'kata_baku']

Jumlah data raw  : 11,163 baris
Kolom            : ['text']


,text
0,presiden terburuk sepanjnag sejarah asli parah
1,nyawit ni orang
2,banyak orang yang pintar ituu di sana tapi sem...
3,siapa yg pilih wowo kemarin
4,ulah si prabowo karena ada dendam pribadi hanc...


In [4]:
# ── Setup Komponen Preprocessing ─────────────────────────────────────────────
print('=' * 55)
print('  SETUP KOMPONEN PREPROCESSING')
print('=' * 55)

stemmer                = setup_stemmer()
stopword_github, _     = load_stopwords_github()
stopword_final         = build_stopword_final(stopword_github)
kamus_gabungan         = build_kamus_gabungan(kamus_excel)

  SETUP KOMPONEN PREPROCESSING
[preprocessing] Stemmer Sastrawi siap.


[preprocessing] Stopword dari GitHub  : 163 kata
[preprocessing] Kata negasi (dijaga)  : 12 kata
[preprocessing] Stopword manual       : 92 kata
[preprocessing] Stopword GitHub       : 163 kata
[preprocessing] Stopword gabungan     : 248 kata
[preprocessing] Stopword final (aman) : 248 kata

[preprocessing] Verifikasi — kata negasi masih ada di stopword?
  'anti      ' → aman
  'belum     ' → aman
  'bukan     ' → aman
  'jangan    ' → aman
  'mustahil  ' → aman
  'non       ' → aman
  'tak       ' → aman
  'tanpa     ' → aman
  'tiada     ' → aman
  'tidak     ' → aman
[preprocessing] Total entri kamus gabungan: 4360


In [5]:
# ── Jalankan Full Preprocessing Pipeline ─────────────────────────────────────
df_clean = run_preprocessing(
    df_raw,
    kamus_gabungan,
    stopword_final,
    stemmer,
    text_col='text'
)

[preprocessing] Menjalankan cleaning pada 10821 baris...


[preprocessing] Cleaning selesai.

  LAPORAN DATA CLEANING
  Baris awal              : 11,163
  Baris duplikat dihapus  : 342
  Baris kosong dihapus    : 50
  Baris 1 kata dihapus    : 476
  Baris akhir (bersih)    : 10,295

  CONTOH HASIL (5 baris pertama):
-------------------------------------------------------
  [1] SEBELUM : presiden terburuk sepanjnag sejarah asli parah
       SESUDAH : presiden buruk sepanjnag sejarah asli parah

  [2] SEBELUM : banyak orang yang pintar ituu di sana tapi semua ikut ikutan bego
       SESUDAH : pintar ikut bego demi jawab

  [3] SEBELUM : siapa yg pilih wowo kemarin
       SESUDAH : pilih prabowo kemarin

  [4] SEBELUM : ulah si prabowo karena ada dendam pribadi hancurkan indo membuat 
       SESUDAH : ulah prabowo dendam pribadi hancur indonesia buat indonesia dalam

  [5] SEBELUM : kalau saya yakin yg maha kuasa cuma beri waktu untuk para korupsi
       SESUDAH : yakin maha kuasa cuma beri waktu para korupsi rasa kuasa dulu bel



In [6]:
# ── Simpan Hasil Cleaning ────────────────────────────────────────────────────
save_processed_data(df_clean[['text', 'text_clean']], 'dataset_clean_pipeline.csv')

print('\n[INFO] Hasil cleaning disimpan ke data/processed/dataset_clean_pipeline.csv')
print('[INFO] Untuk pipeline modeling, digunakan dataset_clean_manual.csv')
print('       (hasil labeling manual yang sudah tersedia di data/processed/)')

df_clean[['text', 'text_clean']].head(10)

[data_loader] Data disimpan ke: D:\Kuliah\Semester 6\Data Mining\Tubes\tugas-besar-datamining-kelompok3\data\processed\dataset_clean_pipeline.csv

[INFO] Hasil cleaning disimpan ke data/processed/dataset_clean_pipeline.csv
[INFO] Untuk pipeline modeling, digunakan dataset_clean_manual.csv
       (hasil labeling manual yang sudah tersedia di data/processed/)


,text,text_clean
0,presiden terburuk sepanjnag sejarah asli parah,presiden buruk sepanjnag sejarah asli parah
1,banyak orang yang pintar ituu di sana tapi sem...,pintar ikut bego demi jawab
2,siapa yg pilih wowo kemarin,pilih prabowo kemarin
3,ulah si prabowo karena ada dendam pribadi hanc...,ulah prabowo dendam pribadi hancur indonesia b...
4,kalau saya yakin yg maha kuasa cuma beri waktu...,yakin maha kuasa cuma beri waktu para korupsi ...
5,seharusnya minta maaf dulu karena gak bisa mim...,harus minta maaf dulu tidak mimpin kasih solus...
6,kalo ga kompeten jd presiden turun aja bro gau...,tidak kompeten presiden turun saja tidak usah ...
7,omon omon itu itu lah,omon omon
8,stresssss jawab ngatakan org desa lagi,stresssss jawab ngatakan desa
9,setuju paknaik dolar itu yang panik orang oran...,tuju paknaik dolar panik kayak pengusahakarena...
